In [3]:
# ============================================================
# CELL 0 – LOCAL ENVIRONMENT SETUP
# ============================================================
import os, sys

# Cấu trúc Local-first: lấy thư mục gốc của project
# Vì notebook nằm trong thư mục notebooks/, nên PROJECT_ROOT là thư mục cha.
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
REPO_PATH = PROJECT_ROOT

# Add src/ to Python path
if os.path.join(PROJECT_ROOT, 'src') not in sys.path:
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

print('✅ Local Environment ready. PROJECT_ROOT:', PROJECT_ROOT)


✅ Local Environment ready. PROJECT_ROOT: d:\000MINHTHONG\Junior - Semester II\ANLPB\FinalProject\hate-speech-detection


In [4]:
# ============================================================
# CELL 1 – TRAIN BASELINE MODEL (ROBUST PATH HANDLING)
# ============================================================
import os
import pandas as pd
import yaml
import json
import joblib
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score

# 1. Load configuration
with open(f"{REPO_PATH}/configs/paths.yaml", 'r') as f:
    paths = yaml.safe_load(f)

def resolve_path(path_obj, default_subpath):
    """
    Trích xuất chuỗi đường dẫn và tự động thay thế {project_root} bằng đường dẫn thật.
    """
    if isinstance(path_obj, dict):
        path_str = path_obj.get('dir', list(path_obj.values())[0])
    elif isinstance(path_obj, str):
        path_str = path_obj
    else:
        path_str = default_subpath
        
    # Xử lý nội suy chuỗi {project_root}
    if "{project_root}" in path_str:
        return path_str.replace("{project_root}", PROJECT_ROOT)
    elif "{drive_root}" in path_str:
        return path_str.replace("{drive_root}", PROJECT_ROOT)
        
    # Nếu là đường dẫn tương đối, nối với PROJECT_ROOT
    return os.path.join(PROJECT_ROOT, path_str)

# Thiết lập đường dẫn tuyệt đối an toàn
processed_dir = os.path.join(PROJECT_ROOT, "data", "processed")
model_dir = os.path.join(PROJECT_ROOT, "models", "baseline")
results_dir = os.path.join(PROJECT_ROOT, "results")

# 2. Load data
train = pd.read_parquet(os.path.join(processed_dir, "train.parquet"))
dev = pd.read_parquet(os.path.join(processed_dir, "dev.parquet"))
print(f"✅ Data loaded from: {processed_dir}")

# 3. Feature Extraction (Char n-grams 2-5)
tfidf = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(2, 5),
    max_features=50000,
    sublinear_tf=True
)
X_train = tfidf.fit_transform(train['text'])
X_dev = tfidf.transform(dev['text'])

# 4. Train Model
# Tự động xử lý mất cân bằng nhãn bằng class_weight='balanced'
model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, train['label_id'])

# 5. Evaluate
y_pred = model.predict(X_dev)
macro_f1 = f1_score(dev['label_id'], y_pred, average='macro')
print(f"\n🚀 Baseline Macro F1 Score: {macro_f1:.4f}")
print("\nClassification Report:\n", classification_report(dev['label_id'], y_pred, target_names=['CLEAN', 'OFFENSIVE', 'HATE']))

# 6. Save Artifacts
os.makedirs(model_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)

joblib.dump(model, os.path.join(model_dir, "tfidf_lr.pkl"))
joblib.dump(tfidf, os.path.join(model_dir, "tfidf_vectorizer.pkl"))

# Lưu báo cáo kết quả
with open(os.path.join(results_dir, "baseline_report.json"), 'w') as f:
    json.dump({"macro_f1": float(macro_f1)}, f, indent=4)

print(f"✅ Baseline artifacts saved successfully to Local Storage.")

✅ Data loaded from: d:\000MINHTHONG\Junior - Semester II\ANLPB\FinalProject\hate-speech-detection\data\processed

🚀 Baseline Macro F1 Score: 0.6167

Classification Report:
               precision    recall  f1-score   support

       CLEAN       0.96      0.84      0.90      2180
   OFFENSIVE       0.36      0.52      0.43       212
        HATE       0.43      0.69      0.53       270

    accuracy                           0.80      2662
   macro avg       0.58      0.69      0.62      2662
weighted avg       0.86      0.80      0.82      2662

✅ Baseline artifacts saved successfully to Local Storage.
